# 09 – Stance labelling, stance baseline and actors (NER)

1. **Stance labelling set:** 60 texts (30 chatbot answers, 30 news chunks), a codebook with an example for each label, sheets for two labellers (20 texts both label, 20 only A, 20 only B), and a kappa script tested on dummy labels.
2. **Stance baseline:** VADER-based and TF-IDF + logistic regression classifiers scored against the human labels with cross-validation. The code is written and tested on dummy data; it runs on real labels once the team has produced them.
3. **Actors:** spaCy named-entity recognition over both corpora, actor names normalised (for example `EU Commission` and `European Commission` become one actor), top-actor tables per source and region.

**Labelling is left to the team.** No stance labels are produced here; the dummy labels used in the tests are never saved.

In [1]:
# Guard: this notebook needs files that are not in the repository (publisher text; see README, "Getting the data").
# It stops here, before anything is read or written, so the committed tables in dataset/analysis/ are never overwritten with partial results.
import json
from pathlib import Path
_needed = [json.load(open("config.json"))[k] for k in ['news_corpus_csv']]
_missing = [p for p in _needed if not Path(p).exists()]
if _missing:
    raise SystemExit("Stopped before changing anything: " + ", ".join(_missing) + " not found. This notebook needs the news corpus, which is publisher text and is not part of the repository or the submission ZIP (README, 'Getting the data'). The tables in dataset/analysis/ are the committed results of the original run.")

In [2]:
import json, re, tempfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from agreement import agreement_report, cohen_kappa

CFG = json.load(open("config.json"))
OUT = Path(CFG["analysis_dir"]); FR = Path("dataset/frames"); DATA = Path("data")
SEED = 42
STANCE = ["for", "against", "mixed", "none"]

## 1. Stance labelling set
**Question the labellers answer:** what is the text's position on *government regulation of AI* (or on the specific rule it discusses)?
`for` (favours regulation or stricter rules) · `against` (opposes them) · `mixed` (weighs both sides, or reports both without favouring one) · `none` (takes no position, or is not about regulation, for example a product launch). The plan names for / against / mixed; `none` is added because many news chunks take no position and forcing a label on them would add noise. Kappa is reported on all four labels and on the three plan labels.

**Design:** stratified sample (chatbot: 5 answers for each model × wording; news: 8 US, 7 UK, 7 EU, 8 Other), seeded. 20 shared texts (10 chatbot, 10 news) go to both labellers; 20 only to A; 20 only to B. Opaque ids, shuffled differently per labeller, no source column, no frame scores. With only 20 shared texts the kappa has a wide interval: say so in the report.

In [3]:
def head_words(text, n=300):
    for k, m in enumerate(re.finditer(r"\S+", text)):
        if k == n - 1:
            return text[:m.end()]
    return text

ans = pd.read_json(CFG["answers_jsonl"], lines=True); ans = ans[ans["chatbot"].isin(["llama3.1:8b", "qwen3.5:9b"])].reset_index(drop=True)
news = pd.read_csv(CFG["news_corpus_csv"], keep_default_na=False)

rng = np.random.default_rng(SEED)
ch = pd.concat([g.sample(5, random_state=SEED) for _, g in ans.groupby(["chatbot", "wording"])])
nw = pd.concat([news[news["region"] == r].sample(n, random_state=SEED) for r, n in {"US": 8, "UK": 7, "EU": 7, "Other": 8}.items()])
ch_items = [{"source": "chatbot", "ref": f"{r.prompt_id}|r{r.replicate}|{r.chatbot}", "text": head_words(r.answer)} for r in ch.itertuples()]
nw_items = [{"source": "news", "ref": r.doc_id, "text": head_words(r.text)} for r in nw.itertuples()]
rng.shuffle(ch_items); rng.shuffle(nw_items)
items = pd.DataFrame(ch_items + nw_items)
items["set"] = (["shared"] * 10 + ["A"] * 10 + ["B"] * 10) * 2          # chatbot rows first, then news rows: 10 shared, 10 A, 10 B in each
ids = rng.permutation(len(items)) + 1
items["stance_id"] = [f"S{i:03d}" for i in ids]
key = items[["stance_id", "source", "ref", "set"]].sort_values("stance_id")
INSTR = ("INSTRUCTIONS (row 0, ignore when scoring): put in the stance column what the text's position is on government regulation of AI: "
         "for, against, mixed or none. Examples and rules: dataset/frames/stance_codebook.md. Label alone; do not discuss with the other labeller until both are finished.")
def sheet_for(who, seed):
    s = items[items["set"].isin(["shared", who])].sample(frac=1, random_state=seed)
    return pd.concat([pd.DataFrame([{"stance_id": "S000", "text": INSTR, "stance": "", "note": ""}]), s[["stance_id", "text"]].assign(stance="", note="")], ignore_index=True)
sheet_for("A", 1).to_csv(DATA / "stance_sheet_A.csv", index=False); sheet_for("B", 2).to_csv(DATA / "stance_sheet_B.csv", index=False)
key.to_csv(DATA / "stance_key.csv", index=False)
print(len(items), "texts;", items["set"].value_counts().to_dict(), "| each sheet has", 40, "texts + 1 instruction row | sources in the shared set:", items[items.set == "shared"]["source"].value_counts().to_dict())

60 texts; {'shared': 20, 'A': 20, 'B': 20} | each sheet has 40 texts + 1 instruction row | sources in the shared set: {'chatbot': 10, 'news': 10}


In [4]:
codebook = '''# Stance codebook

**Unit:** one text (the first 300 words of an answer or article).
**Question:** what is the text's position on **government regulation of AI** (or on the specific rule or measure it discusses)?

| label | use it when | invented example (not from the data) |
|---|---|---|
| for | the text favours rules, oversight or restrictions, or argues they are needed | "Strong rules are needed because unchecked AI systems can harm people, and companies will not police themselves." |
| against | the text opposes rules or argues they would do more harm than good | "Heavy-handed AI rules would stifle innovation and push companies and talent to countries with lighter regulation." |
| mixed | the text weighs both sides, or reports both sides without favouring one, or supports some rules and opposes others | "Rules could protect people from harm, but if they are too strict they risk slowing down small companies; the details matter." |
| none | no position on regulation, or the text is not about regulation | "The company unveiled a new AI assistant with a faster model and a longer battery life." |

**Rules**
1. Label the text's own position, not the topic. A text about a ban is not "for" the ban unless it endorses it.
2. A quoted opinion counts only if the text clearly adopts it. A report that quotes one supporter and one critic is `mixed`.
3. Descriptive reporting of a new law with no evaluation is `none`.
4. If a text argues one side but admits a drawback in a sentence, label the side it argues (`for` or `against`), not `mixed`.
5. Chatbot answers to "why is it a good idea" or "why is it a bad idea" prompts often argue one side: label what the text says, not what the prompt asked.
6. If unsure between a side and `mixed`, choose `mixed`; if unsure between `mixed` and `none`, choose `none` only when regulation is not discussed.
7. Label alone. Compare after both labellers have finished.

**Agreement:** the 20 shared texts give Cohen's kappa (all four labels, and the three labels for / against / mixed). Disputed shared texts are discussed and given one consensus label.
'''
# The codebook was extended after this cell was written (human-review rule, content note), so an existing file is never overwritten:
# rewriting it from this template would silently remove those additions. Delete the file by hand to regenerate the original.
if (FR / "stance_codebook.md").exists():
    print("dataset/frames/stance_codebook.md exists: kept as it is (not overwritten)")
else:
    (FR / "stance_codebook.md").write_text(codebook); print("written: dataset/frames/stance_codebook.md")

dataset/frames/stance_codebook.md exists: kept as it is (not overwritten)


### The kappa script (tested on dummy labels)
`score_stance` reads the two labelled sheets, takes the texts both labelled, and reports observed agreement, Cohen's kappa on four labels and on the three plan labels, the confusion matrix and the disputed texts. `build_consensus` produces one label per text (agreed label for shared texts that match; `DISPUTED` for shared texts to discuss; the single labeller's label for A-only and B-only texts).

In [5]:
def read_sheet(path):
    # returns (labels without skipped texts, set of skipped ids, set of all ids on the sheet); 'skip' = the labeller skipped a distressing text
    s = pd.read_csv(path, keep_default_na=False); s = s[s["stance_id"] != "S000"]
    all_ids = set(s["stance_id"]); skipped = set(s.loc[s["stance"] == "skip", "stance_id"]); s = s[s["stance"] != "skip"]
    bad = sorted(set(s["stance"]) - set(STANCE))
    if bad or (s["stance"] == "").any():
        raise ValueError(f"{path}: empty or invalid labels {bad}")
    return s[["stance_id", "stance"]].set_index("stance_id")["stance"], skipped, all_ids

def score_stance(path_A, path_B, key_path=DATA / "stance_key.csv", save=True):
    (A, skA, idsA), (B, skB, idsB) = read_sheet(path_A), read_sheet(path_B)
    expected = set(pd.read_csv(key_path)[lambda d: d["set"] == "shared"]["stance_id"])
    if idsA & idsB != expected:
        raise ValueError(f"the two sheets share {len(idsA & idsB)} texts, expected {len(expected)} shared texts from the key")
    shared = sorted(set(A.index) & set(B.index))             # shared texts that both labellers labelled
    a, b = A.loc[shared].to_numpy(), B.loc[shared].to_numpy()
    rep4 = agreement_report(a, b, STANCE)
    three = np.isin(a, STANCE[:3]) & np.isin(b, STANCE[:3])
    k3 = cohen_kappa(a[three], b[three], STANCE[:3]) if three.sum() > 1 else float("nan")
    res = {"n_shared": len(shared), "skipped_shared": len(expected) - len(shared), "agreement": rep4["accuracy"], "kappa_4_labels": rep4["kappa"], "n_three_label": int(three.sum()), "kappa_3_labels": k3,
           "confusion": rep4["confusion"], "disputed": [s for s, x, y in zip(shared, a, b) if x != y]}
    if save:
        pd.DataFrame([{k: v for k, v in res.items() if k not in ("confusion", "disputed")}]).round(3).to_csv(OUT / "stance_agreement.csv", index=False)
        res["confusion"].to_csv(OUT / "stance_agreement_confusion.csv")
    return res

def build_consensus(path_A, path_B):
    (A, skA, _), (B, skB, _) = read_sheet(path_A), read_sheet(path_B); rows = {}
    for s in A.index.union(B.index):
        if s in A.index and s in B.index: rows[s] = A[s] if A[s] == B[s] else "DISPUTED"
        else: rows[s] = A[s] if s in A.index else B[s]
    for s in skA & skB: rows[s] = "SKIPPED"                 # skipped by everyone who was asked: no label
    return pd.Series(rows, name="stance")

def landis_koch(k):
    return "poor" if k < 0 else "slight" if k <= .2 else "fair" if k <= .4 else "moderate" if k <= .6 else "substantial" if k <= .8 else "almost perfect"

# ---- tests with DUMMY labels (temporary files, nothing saved in the repo) ----
rngt = np.random.default_rng(SEED); kk = pd.read_csv(DATA / "stance_key.csv"); shared_ids = kk[kk["set"] == "shared"]["stance_id"].tolist()
def dummy_sheet(who, agree_with=None, p=1.0):
    s = pd.read_csv(DATA / f"stance_sheet_{who}.csv", keep_default_na=False)
    labs = []
    for sid in s["stance_id"]:
        if sid == "S000": labs.append("")
        elif agree_with is not None and sid in agree_with and rngt.random() < p: labs.append(agree_with[sid])
        else: labs.append(rngt.choice(STANCE))
    return s.assign(stance=labs)
with tempfile.TemporaryDirectory() as td:
    td = Path(td); sA = dummy_sheet("A"); pA = td / "A.csv"; sA.to_csv(pA, index=False)
    truth = dict(zip(sA["stance_id"], sA["stance"]))
    pB = td / "B_same.csv"; dummy_sheet("B", truth, 1.0).to_csv(pB, index=False)
    r = score_stance(pA, pB, save=False); assert r["agreement"] == 1 and abs(r["kappa_4_labels"] - 1) < 1e-12 and r["disputed"] == [], r
    pB2 = td / "B_random.csv"; dummy_sheet("B").to_csv(pB2, index=False); r2 = score_stance(pA, pB2, save=False)
    assert r2["agreement"] < 0.7, r2
    # an invalid label must stop the script
    bad = dummy_sheet("B", truth, 1.0); bad.loc[3, "stance"] = "maybe"; pbad = td / "B_bad.csv"; bad.to_csv(pbad, index=False)
    try: score_stance(pA, pbad, save=False); raise AssertionError("invalid label was accepted")
    except ValueError: pass
    skip_b = dummy_sheet("B", truth, 1.0); skip_b.loc[skip_b["stance_id"] == shared_ids[0], "stance"] = "skip"; pskip = td / "B_skip.csv"; skip_b.to_csv(pskip, index=False)
    rs = score_stance(pA, pskip, save=False); assert rs["skipped_shared"] == 1 and rs["n_shared"] == 19 and rs["agreement"] == 1, rs
    cons = build_consensus(pA, pB2); assert len(cons) == 60 and (cons == "DISPUTED").sum() == len(r2["disputed"])
print("stance kappa script: identical sheets -> kappa 1; random labellers -> agreement", round(r2["agreement"], 2), "kappa", round(r2["kappa_4_labels"], 2),
      "; invalid label rejected; a skipped shared text is dropped and counted; consensus keeps", int((cons == 'DISPUTED').sum()), "disputed texts for discussion")

stance kappa script: identical sheets -> kappa 1; random labellers -> agreement 0.3 kappa 0.07 ; invalid label rejected; a skipped shared text is dropped and counted; consensus keeps 14 disputed texts for discussion


**To score real labels:** save the two labelled sheets (for example `data/stance_sheet_A_labelled.csv` and `data/stance_sheet_B_labelled.csv`), then run
`res = score_stance("data/stance_sheet_A_labelled.csv", "data/stance_sheet_B_labelled.csv")`, read `res["kappa_4_labels"]`, `res["disputed"]` (texts to discuss), and `landis_koch(kappa)` for the verbal label. After the discussion, write the agreed label for each disputed text into a copy of `build_consensus(...)` and save it as `data/stance_labels_consensus.csv` (columns `stance_id`, `stance`) for the baseline below.

### 1b. Both labellers: agreement on the 20 shared texts, disputed texts, reference labels (2026-10-30)
Both stance sheets are labelled (`data/stance_sheet_A_labelled.csv`, `data/stance_sheet_B_labelled.csv`). `score_stance` gives Cohen's kappa on the 20 shared texts (four labels and the three plan labels); `build_consensus` gives one label per text: the agreed label for shared texts both gave, `DISPUTED` for shared texts they gave differently, and the single labeller's label for the 20 A-only and 20 B-only texts. **Disputed texts are not adjudicated here** (that is a discussion for the two labellers): they are listed by id and left out of the baseline, which is therefore scored against agreed shared labels plus single-labeller labels. The disputed list carries ids, source and the two labels, never text. The consensus file is `data/stance_labels_consensus.csv` (git-ignored).

In [6]:
LAB_A, LAB_B = DATA / "stance_sheet_A_labelled.csv", DATA / "stance_sheet_B_labelled.csv"
if LAB_A.exists() and LAB_B.exists():
    res = score_stance(LAB_A, LAB_B)                                          # writes stance_agreement.csv and stance_agreement_confusion.csv
    keyS = pd.read_csv(DATA / "stance_key.csv").set_index("stance_id")
    A_, _, _ = read_sheet(LAB_A); B_, _, _ = read_sheet(LAB_B)
    shared = sorted(set(A_.index) & set(B_.index))
    print(f"shared texts: {res['n_shared']} | observed agreement {res['agreement']:.2f} | kappa on 4 labels {res['kappa_4_labels']:.3f} ({landis_koch(res['kappa_4_labels'])}) | kappa on the 3 plan labels {res['kappa_3_labels']:.3f} (n={res['n_three_label']})")
    display(res["confusion"])
    rows = [{"scope": "all shared texts", "n": len(shared), "agreement": round(res["agreement"], 3), "kappa_4_labels": round(res["kappa_4_labels"], 3)}]
    for src in ("chatbot", "news"):
        ids = [s for s in shared if keyS.loc[s, "source"] == src]; a, b = A_.loc[ids].to_numpy(), B_.loc[ids].to_numpy()
        rows.append({"scope": f"shared {src} texts", "n": len(ids), "agreement": round(float((a == b).mean()), 3), "kappa_4_labels": round(float(cohen_kappa(a, b, STANCE)), 3)})
    by_src = pd.DataFrame(rows); by_src.to_csv(OUT / "stance_agreement_by_source.csv", index=False); display(by_src)
    # label distribution of each labeller on the shared texts and overall
    dist = pd.DataFrame({"labeller_A_all_40": A_.value_counts(), "labeller_B_all_40": B_.value_counts(), "labeller_A_shared_20": A_.loc[shared].value_counts(), "labeller_B_shared_20": B_.loc[shared].value_counts()}).reindex(STANCE).fillna(0).astype(int)
    dist.to_csv(OUT / "stance_label_counts_by_labeller.csv"); display(dist)
    disputed = pd.DataFrame([{"stance_id": s, "source": keyS.loc[s, "source"], "labeller_A": A_[s], "labeller_B": B_[s]} for s in res["disputed"]])
    disputed.to_csv(OUT / "stance_disputed_texts.csv", index=False); print(len(disputed), "disputed texts (ids only, no text); to adjudicate:", disputed["stance_id"].tolist())
    cons = build_consensus(LAB_A, LAB_B); cons.rename_axis("stance_id").reset_index().to_csv(DATA / "stance_labels_consensus.csv", index=False)
    print("reference labels written to data/stance_labels_consensus.csv:", cons.value_counts().to_dict())
else:
    print("both labelled stance sheets are needed (data/stance_sheet_A_labelled.csv and data/stance_sheet_B_labelled.csv)")

shared texts: 20 | observed agreement 0.45 | kappa on 4 labels 0.254 (fair) | kappa on the 3 plan labels -0.158 (n=11)


checked,for,against,mixed,none
reference,,,,
for,2,1,1,0
against,1,0,0,0
mixed,5,0,1,3
none,0,0,0,6


,scope,n,agreement,kappa_4_labels
0,all shared texts,20,0.45,0.254
1,shared chatbot texts,10,0.30,-0.148
2,shared news texts,10,0.60,0.130


,labeller_A_all_40,labeller_B_all_40,labeller_A_shared_20,labeller_B_shared_20
stance,,,,
for,8,12,4,8
against,5,4,1,1
mixed,12,6,9,2
none,15,18,6,9


11 disputed texts (ids only, no text); to adjudicate: ['S001', 'S004', 'S008', 'S012', 'S029', 'S031', 'S038', 'S050', 'S052', 'S053', 'S058']
reference labels written to data/stance_labels_consensus.csv: {'none': 24, 'DISPUTED': 11, 'for': 10, 'mixed': 8, 'against': 7}


### 1c. Chatbot labels against the prompt wording (a flagging check, not a label source) (2026-10-30)
The 30 chatbot texts come from prompts worded neutral, *leaning for* ("why is it a good idea ...") or *leaning against* ("why is it a bad idea ..."). Each labeller's stance label is cross-tabulated against the wording. For a leaning prompt the expected direction is `for` (leaning for) or `against` (leaning against). A label is a **match** if it equals that direction, a **contradiction** if it is the opposite direction, and **other** if it is `mixed` or `none`. Neutral prompts have no direction.

**This is a flagging check only.** Codebook rule 5 says to label what the text says, not what the prompt asked, so a label that differs from the prompt direction is not wrong by itself: the answer may really argue the other way or weigh both sides. The check only produces a list of texts worth a second look. **It is never used to choose or change a label**, and the adjudication sheet does not show the prompt wording. Files: `stance_vs_prompt_wording.csv` (cross-tabulation), `stance_prompt_direction_summary.csv` (shares), `stance_prompt_direction_flags.csv` (the flagged texts, ids only, no text).

In [7]:
if LAB_A.exists() and LAB_B.exists():
    prompts_df = pd.read_csv(CFG["prompts_csv"]).set_index("prompt_id")
    keyC = pd.read_csv(DATA / "stance_key.csv", keep_default_na=False); chat_k = keyC[keyC["source"] == "chatbot"].copy()
    chat_k["prompt_id"] = chat_k["ref"].str.split("|").str[0]; chat_k["wording"] = chat_k["prompt_id"].map(prompts_df["wording"])
    DIRECTION = {"leaning_for": "for", "leaning_against": "against"}; OPPOSITE = {"for": "against", "against": "for"}
    WORDINGS = ["neutral", "leaning_for", "leaning_against"]
    def relation(wording, label):
        if wording not in DIRECTION: return "no direction (neutral prompt)"
        if label == DIRECTION[wording]: return "match"
        if label == OPPOSITE[DIRECTION[wording]]: return "contradiction"
        return "other (mixed or none)"
    xt_rows, sm_rows, flag_rows = [], [], []
    for nm, lab in (("A", A_), ("B", B_)):
        d = chat_k[chat_k["stance_id"].isin(lab.index)].copy(); d["label"] = lab.loc[d["stance_id"]].to_numpy(); d["relation"] = [relation(w, l) for w, l in zip(d["wording"], d["label"])]
        ct = pd.crosstab(d["wording"], d["label"]).reindex(index=WORDINGS, columns=STANCE, fill_value=0)
        xt_rows.append(ct.reset_index().assign(labeller=nm, scope="all chatbot texts this labeller labelled").assign(n=ct.sum(axis=1).to_numpy()))
        for scope, sub in (("all chatbot texts this labeller labelled", d), ("the 10 chatbot texts both labelled", d[d["set"] == "shared"])):
            lean = sub[sub["wording"].isin(DIRECTION)]; n = len(lean); cnt = lean["relation"].value_counts()
            sm_rows.append({"labeller": nm, "scope": scope, "n_chatbot_texts": len(sub), "n_leaning_prompts": n, "match": int(cnt.get("match", 0)), "contradiction": int(cnt.get("contradiction", 0)),
                            "other_mixed_or_none": int(cnt.get("other (mixed or none)", 0)), "share_match": round(cnt.get("match", 0) / n, 3) if n else np.nan,
                            "share_contradiction": round(cnt.get("contradiction", 0) / n, 3) if n else np.nan, "share_other": round(cnt.get("other (mixed or none)", 0) / n, 3) if n else np.nan})
        for r in d[d["relation"].isin(["contradiction", "other (mixed or none)"])].itertuples():
            flag_rows.append({"stance_id": r.stance_id, "labelled_by": {"shared": "A and B", "A": "A only", "B": "B only"}[r.set], "labeller": nm, "prompt_wording": r.wording,
                              "prompt_direction": DIRECTION[r.wording], "label": r.label, "relation": r.relation})
    xt = pd.concat(xt_rows, ignore_index=True)[["labeller", "scope", "wording", "for", "against", "mixed", "none", "n"]]; xt.to_csv(OUT / "stance_vs_prompt_wording.csv", index=False)
    sm = pd.DataFrame(sm_rows); sm.to_csv(OUT / "stance_prompt_direction_summary.csv", index=False)
    fl = pd.DataFrame(flag_rows).sort_values(["relation", "stance_id", "labeller"]); fl["note"] = "flagging check only; not a label source"
    fl.to_csv(OUT / "stance_prompt_direction_flags.csv", index=False)
    display(xt[xt["scope"].str.startswith("all")]); display(sm); display(fl)
    print("contradictions (opposite of the prompt direction):", fl[fl["relation"] == "contradiction"][["stance_id", "labelled_by", "labeller"]].values.tolist())

label,labeller,scope,wording,for,against,mixed,none,n
0,A,all chatbot texts this labeller labelled,neutral,1,0,3,0,4
1,A,all chatbot texts this labeller labelled,leaning_for,5,1,2,1,9
2,A,all chatbot texts this labeller labelled,leaning_against,2,4,0,1,7
3,B,all chatbot texts this labeller labelled,neutral,5,0,5,0,10
4,B,all chatbot texts this labeller labelled,leaning_for,6,0,0,0,6
5,B,all chatbot texts this labeller labelled,leaning_against,0,4,0,0,4


,labeller,scope,n_chatbot_texts,n_leaning_prompts,match,contradiction,other_mixed_or_none,share_match,share_contradiction,share_other
0,A,all chatbot texts this labeller labelled,20,16,9,3,4,0.562,0.188,0.250
1,A,the 10 chatbot texts both labelled,10,6,2,2,2,0.333,0.333,0.333
2,B,all chatbot texts this labeller labelled,20,10,10,0,0,1.000,0.000,0.000
3,B,the 10 chatbot texts both labelled,10,6,6,0,0,1.000,0.000,0.000


,stance_id,labelled_by,labeller,prompt_wording,prompt_direction,label,relation,note
0,S012,A and B,A,leaning_for,for,against,contradiction,flagging check only; not a label source
2,S029,A and B,A,leaning_against,against,for,contradiction,flagging check only; not a label source
3,S044,A only,A,leaning_against,against,for,contradiction,flagging check only; not a label source
1,S021,A only,A,leaning_for,for,none,other (mixed or none),flagging check only; not a label source
4,S045,A only,A,leaning_against,against,none,other (mixed or none),flagging check only; not a label source
5,S053,A and B,A,leaning_for,for,mixed,other (mixed or none),flagging check only; not a label source
6,S058,A and B,A,leaning_for,for,mixed,other (mixed or none),flagging check only; not a label source


contradictions (opposite of the prompt direction): [['S012', 'A and B', 'A'], ['S029', 'A and B', 'A'], ['S044', 'A only', 'A']]


## 2. Stance baseline (written and tested; runs when consensus labels exist)
Two baselines against the human stance labels, stratified cross-validation, with a majority-class baseline for reference:
- **VADER:** sentence-level compound scores summarised as four features (mean, standard deviation, share of positive and of negative sentences) fed to a logistic regression. VADER measures tone, not stance, so it is expected to be weak: that is the point of the baseline.
- **TF-IDF + logistic regression:** unigrams and bigrams.
The number of folds is the largest of 2–5 that every class can fill; classes with fewer than 2 texts are reported and left out of the cross-validation.

In [8]:
sia = SentimentIntensityAnalyzer()
def vader_features(text):
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    c = np.array([sia.polarity_scores(s)["compound"] for s in sents]) if sents else np.zeros(1)
    return [c.mean(), c.std(), (c > 0.05).mean(), (c < -0.05).mean()]

def stance_baseline(df, seed=SEED):
    """df: columns text, stance. Returns a table of cross-validated accuracy, macro-F1 and kappa vs the human labels."""
    counts = df["stance"].value_counts(); usable = counts[counts >= 2].index
    d = df[df["stance"].isin(usable)].reset_index(drop=True); k = int(min(5, counts[usable].min())); notes = {"left_out_classes": sorted(set(counts.index) - set(usable)), "folds": k}
    if k < 2 or d["stance"].nunique() < 2: raise ValueError("not enough labelled texts per class for cross-validation")
    X_v = np.array([vader_features(t) for t in d["text"]]); y = d["stance"].to_numpy()
    models = {"majority class": None, "VADER features + logistic regression": ("v", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))),
              "TF-IDF (1-2 grams) + logistic regression": ("t", make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000, class_weight="balanced", C=3)))}
    rows = []
    for name, spec in models.items():
        pred = np.empty(len(d), dtype=object)
        for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(d, y):
            if spec is None: pred[te] = pd.Series(y[tr]).mode()[0]; continue
            kind, m = spec; Xtr, Xte = (X_v[tr], X_v[te]) if kind == "v" else (d["text"].iloc[tr], d["text"].iloc[te])
            pred[te] = m.fit(Xtr, y[tr]).predict(Xte)
        rows.append({"model": name, "n": len(d), "accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"), "kappa_vs_human": cohen_kappa(y, pred, sorted(set(y)))})
    return pd.DataFrame(rows).round(3), notes

# ---- tests on DUMMY data: separable synthetic texts, then random labels ----
words = {"for": "support strong rules necessary protect safeguard essential", "against": "oppose burden stifle harmful restrictive unnecessary", "mixed": "however both balance trade-off depends careful",
         "none": "launch product feature release update announced device"}
r0 = np.random.default_rng(1)
synth = pd.DataFrame([{"text": " ".join(r0.choice(words[c].split(), 12)) + ". " + " ".join(r0.choice("the a of and AI new said will".split(), 8)), "stance": c} for c in STANCE for _ in range(15)])
tab, notes = stance_baseline(synth); tf = tab.set_index("model").loc["TF-IDF (1-2 grams) + logistic regression"]
assert tf["accuracy"] > 0.9, tab
shuf = synth.assign(stance=r0.permutation(synth["stance"].to_numpy())); tab2, _ = stance_baseline(shuf)
assert tab2.set_index("model").loc["TF-IDF (1-2 grams) + logistic regression", "kappa_vs_human"] < 0.3, tab2
print("stance baseline tests passed: separable dummy data -> TF-IDF accuracy", tf["accuracy"], "; shuffled labels -> kappa", tab2.set_index("model").loc["TF-IDF (1-2 grams) + logistic regression", "kappa_vs_human"])
tab

stance baseline tests passed: separable dummy data -> TF-IDF accuracy 1.0 ; shuffled labels -> kappa -0.044


,model,n,accuracy,macro_f1,kappa_vs_human
0,majority class,60,0.250,0.100,0.000
1,VADER features + logistic regression,60,0.883,0.885,0.844
2,TF-IDF (1-2 grams) + logistic regression,60,1.000,1.000,1.000


In [9]:
CONSENSUS = DATA / "stance_labels_consensus.csv"
if CONSENSUS.exists():
    lab = pd.read_csv(CONSENSUS, keep_default_na=False); lab = lab[lab["stance"] != "DISPUTED"]
    sheetA, sheetB = pd.read_csv(DATA / "stance_sheet_A.csv", keep_default_na=False), pd.read_csv(DATA / "stance_sheet_B.csv", keep_default_na=False)
    texts = pd.concat([sheetA, sheetB]).drop_duplicates("stance_id").set_index("stance_id")["text"]
    df = lab.assign(text=lab["stance_id"].map(texts)).dropna(subset=["text"])
    df = df[df["stance"] != "SKIPPED"]
    res_tab, notes = stance_baseline(df); res_tab.insert(0, "reference", "agreed shared + single-labeller labels, disputed excluded (n=%d)" % len(df)); res_tab.to_csv(OUT / "stance_baseline.csv", index=False)
    print(notes); display(res_tab)
else:
    print("No", CONSENSUS, "yet: the baseline waits for the human labels (see the instructions above).")

{'left_out_classes': [], 'folds': 5}


,reference,model,n,accuracy,macro_f1,kappa_vs_human
0,"agreed shared + single-labeller labels, disput...",majority class,49,0.490,0.164,0.000
1,"agreed shared + single-labeller labels, disput...",VADER features + logistic regression,49,0.306,0.229,0.041
2,"agreed shared + single-labeller labels, disput...",TF-IDF (1-2 grams) + logistic regression,49,0.592,0.364,0.363


### 2c. The baseline against each labeller separately, and the 20-seed spread (2026-10-30)
The reference above pools agreed labels with single-labeller labels, so it carries both labellers' habits. Here the same classifiers are scored against **each labeller's own 40 labels** and the cross-validation is repeated over 20 seeds, which shows how much the number depends on the labeller and on the random split. Files: `stance_baseline_by_labeller.csv` and `stance_baseline_by_labeller_seeds.csv`.

The label counts by source and labeller (`stance_labels_by_source_and_labeller.csv`) show whether stance is confounded with source in each labeller's labels: if one source carries almost all the for/against labels, a classifier can score above the majority class just by recognising the source.

In [10]:
if LAB_A.exists() and LAB_B.exists():
    textsB = pd.read_csv(DATA / "stance_sheet_B.csv", keep_default_na=False).set_index("stance_id")["text"]; textsA = pd.read_csv(DATA / "stance_sheet_A.csv", keep_default_na=False).set_index("stance_id")["text"]
    keyS2 = pd.read_csv(DATA / "stance_key.csv").set_index("stance_id")
    bysrc = pd.concat([pd.DataFrame({"labeller": nm, "source": keyS2.loc[lab.index, "source"].to_numpy(), "stance": lab.to_numpy()}) for nm, lab in (("A", A_), ("B", B_))])
    bysrc = bysrc.groupby(["labeller", "source", "stance"]).size().unstack(fill_value=0).reindex(columns=STANCE, fill_value=0)
    bysrc.to_csv(OUT / "stance_labels_by_source_and_labeller.csv"); display(bysrc)
    one, spreads = [], []
    for nm, lab, tx in (("labeller A", A_, textsA), ("labeller B", B_, textsB)):
        d1 = pd.DataFrame({"stance_id": lab.index, "stance": lab.to_numpy()}); d1["text"] = d1["stance_id"].map(tx)
        t, _ = stance_baseline(d1); one.append(t.assign(reference=f"{nm} only (n={len(d1)})"))
        r = pd.concat([stance_baseline(d1, seed=s)[0].assign(seed=s) for s in range(20)])
        sp = r.groupby("model")[["accuracy", "macro_f1", "kappa_vs_human"]].agg(["mean", "min", "max"]).round(3); sp.columns = ["_".join(c) for c in sp.columns]
        spreads.append(sp.reset_index().assign(reference=f"{nm} only (n={len(d1)})", seeds=20))
    by_lab = pd.concat(one, ignore_index=True)[["reference", "model", "n", "accuracy", "macro_f1", "kappa_vs_human"]]; by_lab.to_csv(OUT / "stance_baseline_by_labeller.csv", index=False)
    seeds_all = pd.concat(spreads, ignore_index=True); seeds_all.to_csv(OUT / "stance_baseline_by_labeller_seeds.csv", index=False)
    display(by_lab); display(seeds_all[["reference", "model", "kappa_vs_human_mean", "kappa_vs_human_min", "kappa_vs_human_max"]])

stance            for  against  mixed  none
labeller source                            
A        chatbot    8        5      5     2
         news       0        0      7    13
B        chatbot   11        4      5     0
         news       1        0      1    18

,reference,model,n,accuracy,macro_f1,kappa_vs_human
0,labeller A only (n=40),majority class,40,0.375,0.136,0.000
1,labeller A only (n=40),VADER features + logistic regression,40,0.375,0.326,0.147
2,labeller A only (n=40),TF-IDF (1-2 grams) + logistic regression,40,0.475,0.357,0.238
3,labeller B only (n=40),majority class,40,0.450,0.155,0.000
4,labeller B only (n=40),VADER features + logistic regression,40,0.550,0.494,0.364
5,labeller B only (n=40),TF-IDF (1-2 grams) + logistic regression,40,0.650,0.452,0.473


,reference,model,kappa_vs_human_mean,kappa_vs_human_min,kappa_vs_human_max
0,labeller A only (n=40),TF-IDF (1-2 grams) + logistic regression,0.226,0.110,0.357
1,labeller A only (n=40),VADER features + logistic regression,0.026,-0.079,0.159
2,labeller A only (n=40),majority class,0.000,0.000,0.000
3,labeller B only (n=40),TF-IDF (1-2 grams) + logistic regression,0.565,0.423,0.648
4,labeller B only (n=40),VADER features + logistic regression,0.352,0.207,0.458
5,labeller B only (n=40),majority class,0.000,0.000,0.000


### 2d. Rebuilt reference and the three references side by side (runs once a consensus column is filled) (2026-10-30)
**Waiting for the labellers' adjudication.** `data/stance_adjudication.csv` lists the 20 shared texts and the single-labeller chatbot texts that the flagging check marks, with an empty `consensus` column that **only the labellers fill**; this code never fills it and never overwrites the file once it exists. The sheet shows the text and the two labels, not the prompt wording and not the reason a text is listed.

**Rebuilt reference (when the column is filled).** For each text: the `consensus` label if one was written; otherwise the label both labellers agreed on (shared texts), otherwise the single labeller's label (A-only and B-only texts). A shared text that is still disputed and has no consensus is left out, and the count is reported. The baseline is then scored on **three references**: the rebuilt consensus reference, labeller A's own 40 labels, and labeller B's own 40 labels, each with the majority-class baseline in the same table, 20-seed stratified cross-validation, and results for all texts, chatbot texts and news texts separately (accuracy, macro-F1 over the classes present in that subset, Cohen's kappa). Because stance is nearly confounded with source, the separate chatbot and news rows are the fairer reading. Files: `stance_baseline_three_references.csv` (mean, min and max over seeds), `stance_baseline_three_references_seeds.csv`, `stance_reference_label_counts.csv`.

In [11]:
ADJ = DATA / "stance_adjudication.csv"
STANCE_SPECS = {"majority class": None, "VADER features + logistic regression": "v", "TF-IDF (1-2 grams) + logistic regression": "t"}

def baseline_by_source(df, seed=SEED):
    """Stratified CV on df (columns text, stance, source); the same predictions are scored on all texts, chatbot texts and news texts."""
    counts = df["stance"].value_counts(); usable = counts[counts >= 2].index
    d = df[df["stance"].isin(usable)].reset_index(drop=True); k = int(min(5, counts[usable].min()))
    if k < 2 or d["stance"].nunique() < 2: raise ValueError("not enough labelled texts per class for cross-validation")
    X_v = np.array([vader_features(t) for t in d["text"]]); y = d["stance"].to_numpy(); out = []
    for name, kind in STANCE_SPECS.items():
        pred = np.empty(len(d), dtype=object)
        for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(d, y):
            if kind is None: pred[te] = pd.Series(y[tr]).mode()[0]; continue
            m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")) if kind == "v" else \
                make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000, class_weight="balanced", C=3))
            Xtr, Xte = (X_v[tr], X_v[te]) if kind == "v" else (d["text"].iloc[tr], d["text"].iloc[te])
            pred[te] = m.fit(Xtr, y[tr]).predict(Xte)
        for sub, mask in (("all texts", np.ones(len(d), bool)), ("chatbot texts", (d["source"] == "chatbot").to_numpy()), ("news texts", (d["source"] == "news").to_numpy())):
            yt, yp = y[mask], pred[mask]
            if len(yt) == 0: continue
            out.append({"model": name, "subset": sub, "n": int(mask.sum()), "accuracy": accuracy_score(yt, yp), "macro_f1": f1_score(yt, yp, labels=sorted(set(yt)), average="macro", zero_division=0),
                        "kappa_vs_human": cohen_kappa(yt, yp, sorted(set(yt) | set(yp)))})
    return pd.DataFrame(out)

def rebuild_reference(adj_path, labA, labB):
    """consensus where written, else the agreed label, else the single labeller's label; unresolved disputes are left out and counted."""
    adj = pd.read_csv(adj_path, keep_default_na=False); filled = adj[adj["consensus"].str.strip() != ""].set_index("stance_id")["consensus"].str.strip()
    bad = sorted(set(filled) - set(STANCE))
    if bad: raise ValueError(f"invalid consensus labels: {bad} (use one of {STANCE})")
    base = build_consensus(labA, labB); ref = base.copy(); changed = [s for s in filled.index if ref[s] != filled[s]]
    ref.loc[filled.index] = filled
    unresolved = ref[ref.isin(["DISPUTED", "SKIPPED"])].index.tolist()
    return ref.drop(index=unresolved), {"n_consensus_written": len(filled), "n_changed_vs_original": len(changed), "unresolved_disputes_left_out": unresolved}

def run_three_references(adj_path, labA, labB, seeds=20, save=True):
    ref, info = rebuild_reference(adj_path, labA, labB)
    tx = pd.concat([pd.read_csv(DATA / "stance_sheet_A.csv", keep_default_na=False), pd.read_csv(DATA / "stance_sheet_B.csv", keep_default_na=False)]).drop_duplicates("stance_id").set_index("stance_id")["text"]
    ks = pd.read_csv(DATA / "stance_key.csv").set_index("stance_id")["source"]
    A0, _, _ = read_sheet(labA); B0, _, _ = read_sheet(labB)
    refs = {"consensus (rebuilt)": ref, "labeller A only": A0, "labeller B only": B0}; raw, counts = [], []
    for nm, lab in refs.items():
        df = pd.DataFrame({"stance_id": lab.index, "stance": lab.to_numpy()}); df["text"] = df["stance_id"].map(tx); df["source"] = df["stance_id"].map(ks)
        counts.append(df.groupby(["source", "stance"]).size().unstack(fill_value=0).reindex(columns=STANCE, fill_value=0).assign(n=lambda t: t.sum(axis=1)).assign(reference=nm).reset_index())
        for s in range(seeds): raw.append(baseline_by_source(df, seed=s).assign(reference=f"{nm} (n={len(df)})", seed=s))
    raw = pd.concat(raw, ignore_index=True)
    agg = raw.groupby(["reference", "subset", "model"])[["accuracy", "macro_f1", "kappa_vs_human"]].agg(["mean", "min", "max"]).round(3)
    agg.columns = ["_".join(c) for c in agg.columns]; agg = agg.reset_index().merge(raw.groupby(["reference", "subset", "model"])["n"].first().reset_index(), on=["reference", "subset", "model"]); agg["seeds"] = seeds
    agg = agg[["reference", "subset", "model", "n", "seeds"] + [c for c in agg.columns if c not in ("reference", "subset", "model", "n", "seeds")]]
    if save:
        raw.round(4).to_csv(OUT / "stance_baseline_three_references_seeds.csv", index=False); agg.to_csv(OUT / "stance_baseline_three_references.csv", index=False)
        pd.concat(counts, ignore_index=True)[["reference", "source"] + STANCE + ["n"]].to_csv(OUT / "stance_reference_label_counts.csv", index=False)
    return agg, info, pd.concat(counts, ignore_index=True)

# ---- test on DUMMY adjudications in a temporary file (never saved): the code path works end to end
with tempfile.TemporaryDirectory() as td:
    tmp = Path(td) / "adj.csv"; dummy = pd.DataFrame({"stance_id": sorted(set(A_.index) & set(B_.index)), "consensus": ""})
    dummy.loc[:4, "consensus"] = [A_[s] for s in dummy.loc[:4, "stance_id"]]                       # five dummy adjudications (labeller A's label)
    dummy.to_csv(tmp, index=False)
    agg_t, info_t, cnt_t = run_three_references(tmp, LAB_A, LAB_B, seeds=2, save=False)
    assert info_t["n_consensus_written"] == 5 and set(agg_t["subset"]) == {"all texts", "chatbot texts", "news texts"} and agg_t["reference"].nunique() == 3
    assert (agg_t.groupby(["reference", "subset"])["model"].nunique() == 3).all(), "majority class, VADER and TF-IDF must all be in every block"
    try:
        bad = dummy.copy(); bad.loc[0, "consensus"] = "yes"; bad.to_csv(tmp, index=False); rebuild_reference(tmp, LAB_A, LAB_B); raise SystemExit("an invalid label was accepted")
    except ValueError: pass
print("2d: dummy adjudications -> three references x three subsets x three models built; an invalid consensus label is rejected (nothing saved)")

# ---- the real run, only when someone has filled the consensus column
if not ADJ.exists():
    shared_ids = sorted(set(A_.index) & set(B_.index))
    flagged_single = sorted(set(fl.loc[fl["labelled_by"].isin(["A only", "B only"]), "stance_id"])) if "fl" in globals() else []
    tx = pd.concat([pd.read_csv(DATA / "stance_sheet_A.csv", keep_default_na=False), pd.read_csv(DATA / "stance_sheet_B.csv", keep_default_na=False)]).drop_duplicates("stance_id").set_index("stance_id")["text"]
    ids = sorted(set(shared_ids) | set(flagged_single))
    pd.DataFrame({"stance_id": ids, "text": [tx[s] for s in ids], "labeller_A": [A_.get(s, "") for s in ids], "labeller_B": [B_.get(s, "") for s in ids], "consensus": ""}).to_csv(ADJ, index=False)
    print("wrote the empty adjudication sheet", ADJ, "with", len(ids), "texts (20 shared +", len(flagged_single), "flagged single-labeller texts); consensus is left empty for the labellers")
adj_now = pd.read_csv(ADJ, keep_default_na=False)
if (adj_now["consensus"].str.strip() == "").all():
    print("consensus column is empty: the rebuilt reference and the three-reference baseline are NOT computed yet (no files written)")
else:
    agg, info, cnts = run_three_references(ADJ, LAB_A, LAB_B); print(info); display(cnts); display(agg)

2d: dummy adjudications -> three references x three subsets x three models built; an invalid consensus label is rejected (nothing saved)


{'n_consensus_written': 16, 'n_changed_vs_original': 13, 'unresolved_disputes_left_out': []}


stance,source,for,against,mixed,none,n,reference
0,chatbot,14,7,7,2,30,consensus (rebuilt)
1,news,1,0,3,26,30,consensus (rebuilt)
2,chatbot,8,5,5,2,20,labeller A only
3,news,0,0,7,13,20,labeller A only
4,chatbot,11,4,5,0,20,labeller B only
5,news,1,0,1,18,20,labeller B only


,reference,subset,model,n,seeds,accuracy_mean,accuracy_min,accuracy_max,macro_f1_mean,macro_f1_min,macro_f1_max,kappa_vs_human_mean,kappa_vs_human_min,kappa_vs_human_max
0,consensus (rebuilt) (n=60),all texts,TF-IDF (1-2 grams) + logistic regression,60,20,0.672,0.650,0.700,0.463,0.391,0.521,0.500,0.462,0.543
1,consensus (rebuilt) (n=60),all texts,VADER features + logistic regression,60,20,0.422,0.350,0.500,0.338,0.263,0.395,0.190,0.095,0.283
2,consensus (rebuilt) (n=60),all texts,majority class,60,20,0.467,0.467,0.467,0.159,0.159,0.159,0.000,0.000,0.000
3,consensus (rebuilt) (n=60),chatbot texts,TF-IDF (1-2 grams) + logistic regression,30,20,0.478,0.433,0.533,0.258,0.171,0.322,0.147,0.023,0.227
4,consensus (rebuilt) (n=60),chatbot texts,VADER features + logistic regression,30,20,0.322,0.233,0.400,0.216,0.121,0.276,0.014,-0.131,0.118
5,consensus (rebuilt) (n=60),chatbot texts,majority class,30,20,0.067,0.067,0.067,0.031,0.031,0.031,0.000,0.000,0.000
6,consensus (rebuilt) (n=60),news texts,TF-IDF (1-2 grams) + logistic regression,30,20,0.867,0.867,0.867,0.310,0.310,0.310,0.000,0.000,0.000
7,consensus (rebuilt) (n=60),news texts,VADER features + logistic regression,30,20,0.523,0.400,0.633,0.310,0.195,0.372,0.017,-0.104,0.091
8,consensus (rebuilt) (n=60),news texts,majority class,30,20,0.867,0.867,0.867,0.310,0.310,0.310,0.000,0.000,0.000
9,labeller A only (n=40),all texts,TF-IDF (1-2 grams) + logistic regression,40,20,0.461,0.375,0.550,0.351,0.285,0.439,0.226,0.110,0.357


### 2e. Adjudication outcome table (no text)
What the labellers decided, as ids and labels only, so the documents can cite it without the git-ignored sheet. `final_label` is the consensus where one was written, otherwise the agreed label, otherwise the single labeller's label (a blank consensus on a single-labeller text keeps that labeller's label).

In [12]:
adj_o = pd.read_csv(ADJ, keep_default_na=False)
if (adj_o["consensus"].str.strip() != "").any():
    ksrc = pd.read_csv(DATA / "stance_key.csv").set_index("stance_id")["source"]; ids = adj_o["stance_id"].tolist(); cons_ = dict(zip(adj_o["stance_id"], adj_o["consensus"].str.strip())); reasons_ = dict(zip(adj_o["stance_id"], adj_o["reason"].str.strip())) if "reason" in adj_o.columns else {}
    rows_ = []
    for s in ids:
        a, b, c = A_.get(s, ""), B_.get(s, ""), cons_[s]
        final = c or (a if a == b else (a or b))
        if a and b: kind = "shared, labels differ" if a != b else "shared, labels agree"; how = ("consensus equals A" if c == a else "consensus equals B" if c == b else "consensus is neither labeller's label") if c else "agreed label kept"
        else: kind = "single label (A only)" if a else "single label (B only)"; how = ("changed by consensus" if c and c != (a or b) else "confirmed by consensus" if c else "no consensus written: the labeller's label kept")
        rows_.append({"stance_id": s, "source": ksrc[s], "labelled_by": "A and B" if a and b else "A only" if a else "B only", "kind": kind, "labeller_A": a, "labeller_B": b, "consensus": c, "final_label": final, "outcome": how, "reason": reasons_.get(s, "")})
    outc = pd.DataFrame(rows_); outc.to_csv(OUT / "stance_adjudication_outcome.csv", index=False)
    display(outc.groupby(["kind", "outcome"]).size().to_frame("texts"))
    assert (outc["final_label"] != "").all()
else:
    print("no consensus written yet: no outcome table")

texts
kind                  outcome                                     
shared, labels agree  agreed label kept                          9
shared, labels differ consensus equals B                        10
                      consensus is neither labeller's label      1
single label (A only) changed by consensus                       2
                      confirmed by consensus                     3

## 3. Actors: named entities with normalised names
spaCy `en_core_web_sm` named-entity recognition on the first 300 words of every text (the same text the frame scorer sees). Actors are `ORG`, `PERSON` and `GPE` entities. Names are normalised to one form (case, a leading *the*, possessive *'s*, then an alias table: `EU Commission` = `European Commission`, `Trump` = `Donald Trump`, `U.S.` = `United States`, and so on). AI product names that the recogniser tags as organisations (`ChatGPT`, `Gemini`, `Claude`, `Grok`) are dropped because they are products, not actors. The counts are **document frequency** (how many texts mention the actor), which is not inflated by one article repeating a name.

In [13]:
ner = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer", "attribute_ruler"])

ALIAS = {
 "eu": "European Union", "e.u.": "European Union", "european union": "European Union", "brussels": "European Union",
 "eu commission": "European Commission", "european commission": "European Commission", "e.u. commission": "European Commission",
 "european parliament": "European Parliament", "eu parliament": "European Parliament",
 "us": "United States", "u.s.": "United States", "u.s": "United States", "usa": "United States", "united states": "United States", "america": "United States", "united states of america": "United States",
 "uk": "United Kingdom", "u.k.": "United Kingdom", "united kingdom": "United Kingdom", "britain": "United Kingdom", "great britain": "United Kingdom",
 "openai": "OpenAI", "open ai": "OpenAI", "google": "Google", "alphabet": "Google", "google deepmind": "Google", "deepmind": "Google",
 "meta": "Meta", "meta platforms": "Meta", "facebook": "Meta", "microsoft": "Microsoft", "anthropic": "Anthropic", "xai": "xAI", "x.ai": "xAI",
 "nvidia": "Nvidia", "apple": "Apple", "amazon": "Amazon", "tesla": "Tesla", "spacex": "SpaceX",
 "white house": "White House", "congress": "US Congress", "us congress": "US Congress", "u.s. congress": "US Congress", "senate": "US Senate", "us senate": "US Senate", "u.s. senate": "US Senate",
 "pentagon": "Pentagon", "department of defense": "Pentagon", "defense department": "Pentagon", "department of war": "Pentagon",
 "ftc": "Federal Trade Commission", "federal trade commission": "Federal Trade Commission", "un": "United Nations", "united nations": "United Nations",
 "trump": "Donald Trump", "donald trump": "Donald Trump", "musk": "Elon Musk", "elon musk": "Elon Musk", "altman": "Sam Altman", "sam altman": "Sam Altman",
 "biden": "Joe Biden", "joe biden": "Joe Biden", "amodei": "Dario Amodei", "dario amodei": "Dario Amodei", "zuckerberg": "Mark Zuckerberg", "mark zuckerberg": "Mark Zuckerberg",
 "huang": "Jensen Huang", "jensen huang": "Jensen Huang", "starmer": "Keir Starmer", "keir starmer": "Keir Starmer", "sunak": "Rishi Sunak", "rishi sunak": "Rishi Sunak",
 "macron": "Emmanuel Macron", "emmanuel macron": "Emmanuel Macron", "china": "China", "beijing": "China",
 # product names that carry the company name belong to the company
 "google gemini": "Google", "google ai": "Google", "microsoft copilot": "Microsoft", "microsoft 365": "Microsoft", "meta ai": "Meta", "amazon web services": "Amazon", "aws": "Amazon",
 "openai’s chatgpt": "OpenAI", "associated press": "Associated Press", "ap": "Associated Press", "the associated press": "Associated Press",
}
PRODUCTS = {"ai", "gpt", "chatgpt", "gemini", "claude", "grok", "copilot", "llama", "gen ai", "genai", "sora", "openclaw", "siri", "a.i.", "llm", "llms"}
STATE = {"European Union", "European Commission", "European Parliament", "United States", "United Kingdom", "China", "White House", "US Congress", "US Senate", "Pentagon", "Federal Trade Commission", "United Nations"}
MEDIA = {"Reuters", "Associated Press", "Bloomberg", "CNN", "BBC", "Fox News", "The New York Times", "New York Times", "Washington Post", "Financial Times", "The Guardian", "Wall Street Journal", "Business Insider", "NBC News", "CNBC", "The Verge", "TechCrunch", "AFP"}
COMPANY = {"OpenAI", "Google", "Meta", "Microsoft", "Anthropic", "xAI", "Nvidia", "Apple", "Amazon", "Tesla", "SpaceX", "DeepSeek", "Samsung", "IBM", "Oracle", "Snowflake", "Salesforce", "Perplexity"}

def normalise_actor(text, label=None):
    t = re.sub(r"[’']s$", "", text.strip()); t = re.sub(r"^(the|The)\s+", "", t).strip(" .,;:\"“”()")
    key = re.sub(r"\s+", " ", t.lower())
    if not key or key in PRODUCTS or re.fullmatch(r"[\d\W]+", key) or len(key) < 2: return None
    return ALIAS.get(key, re.sub(r"\s+", " ", t))

def category(actor, label):
    return "state / regulator" if actor in STATE else "company" if actor in COMPANY else "news media" if actor in MEDIA else {"PERSON": "person", "GPE": "country / place"}.get(label, "other organisation")

# tests
assert normalise_actor("EU Commission") == normalise_actor("European Commission") == normalise_actor("the European Commission") == "European Commission"
assert normalise_actor("U.S.") == normalise_actor("US") == normalise_actor("the United States") == "United States"
assert normalise_actor("Trump") == normalise_actor("Donald Trump") == normalise_actor("Trump’s") == "Donald Trump"
assert normalise_actor("Google Gemini") == normalise_actor("Google") == "Google" and normalise_actor("Microsoft Copilot") == "Microsoft"
assert normalise_actor("ChatGPT") is None and normalise_actor("AI") is None and normalise_actor("2026") is None and normalise_actor("OpenAI") == "OpenAI"
print("actor normalisation tests passed")

actor normalisation tests passed


In [14]:
KEEP = {"ORG", "PERSON", "GPE"}
def actors(texts):
    out = []
    for doc in ner.pipe(texts, batch_size=32):
        ents = []
        for e in doc.ents:
            if e.label_ in KEEP:
                a = normalise_actor(e.text, e.label_)
                if a: ents.append((a, e.label_))
        out.append(ents)
    return out

corp = pd.concat([ans.assign(group=ans["chatbot"].map({"llama3.1:8b": "chatbot: Llama", "qwen3.5:9b": "chatbot: Qwen"}), text_cut=ans["answer"].map(lambda t: " ".join(t.split()[:300]))),
                  news.assign(group="news: " + news["region"], text_cut=news["text"].map(lambda t: " ".join(t.split()[:300])))], ignore_index=True)
corp["ents"] = actors(corp["text_cut"].tolist())
rows = []
for g, d in list(corp.groupby("group")) + [("news: all", corp[corp["group"].str.startswith("news")]), ("chatbot: both", corp[corp["group"].str.startswith("chatbot")])]:
    docs, ments, lab = Counter(), Counter(), {}
    for ents in d["ents"]:
        for a in {x for x, _ in ents}: docs[a] += 1
        for a, l in ents: ments[a] += 1; lab.setdefault(a, Counter())[l] += 1
    for rank, (a, n) in enumerate(sorted(docs.items(), key=lambda kv: (-kv[1], kv[0]))[:20], 1):      # ties broken alphabetically, so the table does not depend on hash order
        rows.append({"source": g, "rank": rank, "actor": a, "category": category(a, lab[a].most_common(1)[0][0]), "texts_mentioning": n,
                     "share_of_texts_%": round(100 * n / len(d), 1), "mentions": ments[a], "n_texts_in_source": len(d)})
top = pd.DataFrame(rows); top.to_csv(OUT / "actors_top_by_source.csv", index=False)
top[top["source"].isin(["news: all", "chatbot: both"])].pivot_table(index="rank", columns="source", values="actor", aggfunc="first")

source,chatbot: both,news: all
rank,,
1,Google,United States
2,Amazon,Google
3,China,OpenAI
4,European Union,Microsoft
5,Microsoft,United Kingdom
6,Stricter AI,Anthropic
7,Alexa,Donald Trump
8,Governmental,Elon Musk
9,Healthcare,California


In [15]:
# who is talked about: share of actor mentions by category, and entities per text
def cat_share(d):
    c = Counter(category(a, l) for ents in d["ents"] for a, l in ents); t = sum(c.values()) or 1
    return {**{k: round(100 * v / t, 1) for k, v in c.items()}, "entities_per_text": round(sum(len(e) for e in d["ents"]) / len(d), 1), "texts": len(d)}
shares = pd.DataFrame({g: cat_share(d) for g, d in corp.groupby("group")}).T.fillna(0)
shares.to_csv(OUT / "actor_category_shares.csv"); shares

,state / regulator,person,company,other organisation,entities_per_text,texts,country / place,news media
chatbot: Llama,26.7,26.7,33.3,13.3,0.1,108.0,0.0,0.0
chatbot: Qwen,25.0,0.0,50.0,25.0,0.0,108.0,0.0,0.0
news: EU,9.8,25.8,11.7,37.0,13.5,82.0,14.5,1.3
news: Other,2.3,23.3,13.2,48.4,13.5,104.0,11.5,1.3
news: UK,10.9,27.8,8.0,41.3,12.7,101.0,10.4,1.6
news: US,11.7,26.1,13.5,36.1,14.1,133.0,10.8,1.8


In [16]:
# which spellings were merged? (raw strings per normalised actor, for the actors with the most variants)
var = defaultdict(Counter)
for doc in ner.pipe(corp["text_cut"].tolist()[:], batch_size=32):
    for e in doc.ents:
        if e.label_ in KEEP:
            a = normalise_actor(e.text, e.label_)
            if a: var[a][e.text.strip()] += 1
merged = pd.DataFrame([{"actor": a, "raw_spellings": ", ".join(f"{s} ({n})" for s, n in c.most_common(6)), "n_spellings": len(c)} for a, c in var.items() if len(c) > 1]).sort_values("n_spellings", ascending=False)
merged.head(12)

,actor,raw_spellings,n_spellings
1,European Union,"EU (25), the European Union's (3), the Europea...",8
0,United States,"US (86), U.S. (65), the United States (22), Am...",6
8,Pentagon,"Pentagon (54), the Department of War (7), the ...",6
10,Meta,"Meta (36), Meta AI (13), Meta Platforms (3), M...",5
3,Google,"Google (98), Google Gemini (23), Google DeepMi...",5
7,Donald Trump,"Trump (56), Donald Trump (28), Trump’s (3), Tr...",5
26,Associated Press,"AP (10), The Associated Press (4), Associated ...",4
24,White House,"White House (6), The White House (4), the Whit...",4
17,United Kingdom,"UK (98), Britain (16), the United Kingdom (5),...",4
12,European Commission,"The European Commission (4), the European Comm...",4


## Limits to state in the report
- The recogniser is a small general-English model (`en_core_web_sm`); it mislabels some entities and misses others. The alias table covers the most frequent actors only; spellings it does not know stay separate.
- Actor tables use the first 300 words of each text, the same text the frame scorer sees, so actors named later in an article are not counted.
- Chatbot answers name almost no entities (about 0.1 per answer for Llama, 0.0 for Qwen) because the prompts are generic policy questions; the few that appear include recogniser errors (for example `Stricter AI`, `Governmental`, `Healthcare`), so the chatbot-vs-news actor comparison says little beyond that chatbots do not name actors.
- Stance labelling has only 20 shared texts, so kappa has a wide interval; two labellers from the same team may share assumptions.